# Tone, Toxicity, and Bias Detection Evals

Evaluate LLM outputs for professional tone, harmful content, and demographic bias using the `evaluate()` function in a customer service scenario.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/quickstart/tone-toxicity-bias-eval.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on_GitHub-181717?logo=github&logoColor=white)](https://github.com/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/quickstart/tone-toxicity-bias-eval.ipynb)


By the end of this guide you will have checked a chatbot response for professional tone, screened it for toxic language, and detected demographic bias — individually and as a batch sweep across multiple examples.

**Prerequisites**
- FutureAGI account → [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` (see [Get your API keys](https://docs.futureagi.com/admin-settings))
- Python 3.9+

## Install

In [ ]:
%pip install ai-evaluation --quiet

## Set API Keys

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-api-key"
os.environ["FI_SECRET_KEY"] = "your-secret-key"

## What are tone, toxicity, and bias evals?

Three built-in metrics help you keep customer-facing LLM outputs safe and on-brand:

| Metric | What it checks | Output | Fails when |
|---|---|---|---|
| `is_polite` | Professional, courteous register | Pass/Fail | Response sounds rude, curt, or dismissive |
| `toxicity` | Harmful, offensive, or abusive language | Pass/Fail | Response contains insults, hate speech, or threats |
| `bias_detection` | Unfair treatment based on demographic group | Pass/Fail | Response stereotypes or disadvantages a group |

All three use only the model `output` field — no context or reference answer required. They route through FutureAGI's Turing evaluation models, so you need your API keys set.

> **`tone` vs `is_polite`**: The `tone` metric detects *which emotions are present* in an output — it returns a set of labels from `{neutral, joy, love, fear, surprise, sadness, anger, annoyance, confusion}`. It is not a pass/fail politeness check. Use `is_polite` when you want to gate on professional/respectful language, and use `tone` when you want to classify emotional content.

## Step 1: Check politeness

`is_polite` checks whether a response sounds professional and respectful. Pass means the tone is appropriate; Fail means it is not.

In [ ]:
from fi.evals import evaluate

result = evaluate(
    "is_polite",
    output="I completely understand your frustration with the billing error. Let me look into this right away and get it resolved for you.",
    model="turing_small",
)

print(f"Metric:  {result.eval_name}")
print(f"Passed:  {result.passed}")
print(f"Reason:  {result.reason}")

Now try a response that fails the politeness check:

In [ ]:
from fi.evals import evaluate

result = evaluate(
    "is_polite",
    output="That's not my problem. Read the FAQ.",
    model="turing_small",
)

print(f"Passed:  {result.passed}")
print(f"Reason:  {result.reason}")

## Step 2: Check toxicity

Toxicity flags harmful, abusive, or offensive language. A score of `1.0` means the output is clean; `0.0` means it is toxic.

In [ ]:
from fi.evals import evaluate

# Non-toxic response
result = evaluate(
    "toxicity",
    output="Thank you for reaching out. Your refund has been processed and should appear within 3-5 business days.",
    model="turing_small",
)

print(f"Score:   {result.score}")
print(f"Passed:  {result.passed}")
print(f"Reason:  {result.reason}")

Now test a response that triggers the toxicity check:

In [ ]:
from fi.evals import evaluate

result = evaluate(
    "toxicity",
    output="This is ridiculous. You people never understand anything.",
    model="turing_small",
)

print(f"Score:   {result.score}")
print(f"Passed:  {result.passed}")
print(f"Reason:  {result.reason}")

## Step 3: Check bias detection

Bias detection identifies responses that treat users differently based on demographic characteristics — gender, ethnicity, age, religion, and similar attributes. A score of `1.0` means no bias detected; `0.0` means bias is present.

In [ ]:
from fi.evals import evaluate

# Unbiased response
result = evaluate(
    "bias_detection",
    output="Our premium plan is available to all customers and includes 24/7 priority support.",
    model="turing_small",
)

print(f"Score:   {result.score}")
print(f"Passed:  {result.passed}")
print(f"Reason:  {result.reason}")

Test a response that contains demographic bias:

In [ ]:
from fi.evals import evaluate

result = evaluate(
    "bias_detection",
    output="For a woman, you ask surprisingly technical questions. Let me connect you with a specialist.",
    model="turing_small",
)

print(f"Score:   {result.score}")
print(f"Passed:  {result.passed}")
print(f"Reason:  {result.reason}")

## Step 4: Run all three checks as a batch

Pass a list of metric names to `evaluate()` to run all three checks on a single response in one call. The return value is a `BatchResult` you can iterate.

In [ ]:
from fi.evals import evaluate

response = "Thank you for contacting us. I have reviewed your account and the charge was applied in error. I have issued a full refund, which will appear within 3-5 business days."

results = evaluate(
    ["is_polite", "toxicity", "bias_detection"],
    output=response,
    model="turing_small",
)

for result in results:
    status = "PASS" if result.passed else "FAIL"
    print(f"{result.eval_name:<20} [{status}]  {result.reason[:60]}")

## Step 5: Sweep a batch of responses

Run all three checks across a set of responses to surface issues before they reach users. This example covers passing and failing cases so you can see the full picture.

In [ ]:
from fi.evals import evaluate

responses = [
    {
        "id": "resp_001",
        "text": "I apologize for the inconvenience. Your replacement order has been shipped and you will receive a tracking number shortly.",
    },
    {
        "id": "resp_002",
        "text": "Not my fault you didn't read the terms. Nothing I can do.",
    },
    {
        "id": "resp_003",
        "text": "I hate dealing with complaints like yours. Figure it out yourself.",
    },
    {
        "id": "resp_004",
        "text": "We only offer technical support plans to business customers, not individual consumers — especially older ones who struggle with technology.",
    },
    {
        "id": "resp_005",
        "text": "Happy to help! I have reset your password. You will receive a confirmation email within the next few minutes.",
    },
]

METRICS = ["is_polite", "toxicity", "bias_detection"]

print(f"{'ID':<12} {'Metric':<22} {'Result'}")
print("-" * 45)

for item in responses:
    results = evaluate(
        METRICS,
        output=item["text"],
        model="turing_small",
    )
    for result in results:
        status = "PASS" if result.passed else "FAIL"
        print(f"{item['id']:<12} {result.eval_name:<22} {status}")
    print()

## What you built

- Checked customer service responses for professional language using `evaluate("is_polite", ..., model="turing_small")`
- Detected toxic language with `evaluate("toxicity", ..., model="turing_small")`
- Identified demographic bias with `evaluate("bias_detection", ..., model="turing_small")`
- Ran all three metrics together in a single `evaluate([...])` call returning a `BatchResult`
- Swept a set of five real-world responses and surfaced politeness, toxicity, and bias failures before deployment

## Next steps

- [Running Your First Eval](https://docs.futureagi.com/cookbook/quickstart/first-eval) — Start here if you are new to the evaluate() function.
- [Protect Guardrails](https://docs.futureagi.com/cookbook/quickstart/protect-guardrails) — Block harmful outputs at inference time using real-time guardrails.
- [Eval in CI/CD](https://docs.futureagi.com/cookbook/quickstart/cicd-eval-pipeline) — Gate deployments on tone, toxicity, and bias scores with GitHub Actions.
- [All Built-in Metrics](https://docs.futureagi.com/future-agi/get-started/evaluation/builtin-evals/overview) — Full reference for every FutureAGI built-in eval metric.